In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución por renta media y mediana a nivel de sección censal

Se han estructurado los datos en la carpeta de inputs para la dimensión socioeconómica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del atlas de distribución de renta de los hogares:
https://www.ine.es/dynt3/inebase/index.htm?padre=12385&capsel=12384

In [2]:
path = os.path.join(DATA_INPUTS_DS, "Indicadores de renta media y mediana")

indicadores = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(indicadores.info())
indicadores.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 192834 entries, 108 to 441935
Data columns (total 7 columns):
 #   Column                                Non-Null Count   Dtype 
---  ------                                --------------   ----- 
 0   Municipios                            192834 non-null  object
 1   Distritos                             192834 non-null  object
 2   Secciones                             192834 non-null  object
 3   Indicadores de renta media y mediana  192834 non-null  object
 4   Periodo                               192834 non-null  int64 
 5   Total                                 164394 non-null  object
 6   Provincia                             192834 non-null  object
dtypes: int64(1), object(6)
memory usage: 11.8+ MB
None


,Municipios,Distritos,Secciones,Indicadores de renta media y mediana,Periodo,Total,Provincia
378911,47186 Valladolid,4718605 Valladolid distrito 05,4718605004 Valladolid sección 05004,Renta bruta media por hogar,2021,52.190,Valladolid
387323,47186 Valladolid,4718610 Valladolid distrito 10,4718610054 Valladolid sección 10054,Mediana de la renta por unidad de consumo,2015,20.650,Valladolid
155024,24170 Torre del Bierzo,2417001 Torre del Bierzo distrito 01,2417001001 Torre del Bierzo sección 01001,Renta bruta media por persona,2015,13.409,Leon
34935,05210 San Juan de la Encinilla,0521001 San Juan de la Encinilla distrito 01,0521001001 San Juan de la Encinilla sección 01001,Renta bruta media por hogar,2017,NaN,Avila
218785,37107 Ciudad Rodrigo,3710701 Ciudad Rodrigo distrito 01,3710701001 Ciudad Rodrigo sección 01001,Mediana de la renta por unidad de consumo,2019,17.150,Salamanca


# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [3]:
indicadores_estandarizados = estandarizar_df_ine(indicadores, "Indicadores de renta media y mediana")
print(indicadores_estandarizados.info())
indicadores_estandarizados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 192834 entries, 108 to 441935
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Provincia  192834 non-null  object
 1   CMuni      192834 non-null  object
 2   CUSEC      192834 non-null  object
 3   Indicador  192834 non-null  object
 4   Periodo    192834 non-null  int64 
 5   Total      164394 non-null  object
dtypes: int64(1), object(5)
memory usage: 10.3+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
31813,Avila,05189,0518901001,Renta neta media por persona,2016,7.556
363540,Valladolid,47121,4712101001,Renta neta media por hogar,2020,22.783
331979,Soria,42166,4216601001,Renta bruta media por persona,2018,NaN
432811,Zamora,49234,4923401001,Renta neta media por persona,2022,11.763
163577,Leon,24222,2422201002,Renta neta media por hogar,2021,28.606


In [4]:
indicadores_estandarizados = clean_total_column(indicadores_estandarizados)

## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [5]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(indicadores_estandarizados)

# Filtro por los años 2021 - 2023.
indicadores_recientes = indicadores_estandarizados[
    indicadores_estandarizados["Periodo"].isin([2021, 2022, 2023])
].copy()

📅 Años disponibles:
[2023 2022 2021 2020 2019 2018 2017 2016 2015]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Renta neta media por persona
  - Renta neta media por hogar
  - Media de la renta por unidad de consumo
  - Mediana de la renta por unidad de consumo
  - Renta bruta media por persona
  - Renta bruta media por hogar
------------------------------------------------------------


In [6]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
indicadores_por_seccion = pivotar_indicadores(indicadores_recientes)
print(indicadores_por_seccion.info())
indicadores_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10550 entries, 0 to 10549
Data columns (total 10 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Provincia                                  10550 non-null  object
 1   CMuni                                      10550 non-null  object
 2   CUSEC                                      10550 non-null  object
 3   Periodo                                    10550 non-null  int64 
 4   Media_de_la_renta_por_unidad_de_consumo    8390 non-null   Int64 
 5   Mediana_de_la_renta_por_unidad_de_consumo  8390 non-null   Int64 
 6   Renta_bruta_media_por_hogar                10550 non-null  Int64 
 7   Renta_bruta_media_por_persona              10550 non-null  Int64 
 8   Renta_neta_media_por_hogar                 10550 non-null  Int64 
 9   Renta_neta_media_por_persona               10550 non-null  Int64 
dtypes: Int64(6), int64(1), object(3)
m

,Provincia,CMuni,CUSEC,Periodo,Media_de_la_renta_por_unidad_de_consumo,Mediana_de_la_renta_por_unidad_de_consumo,Renta_bruta_media_por_hogar,Renta_bruta_media_por_persona,Renta_neta_media_por_hogar,Renta_neta_media_por_persona
3927,Leon,24214,2421401001,2022,18296,16450,30205,15172,26548,13335
3605,Leon,24142,2414201002,2021,19312,17850,37670,15731,31302,13072
8045,Valladolid,47032,4703201001,2021,25666,21350,53412,22398,42318,17746
4330,Palencia,34112,3411201001,2023,<NA>,<NA>,41245,20089,34493,16801
3616,Leon,24142,2414202002,2023,18606,17850,35117,15145,29958,12920


# Export de los resultados

In [8]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "rentas_por_seccion.csv")

# Guardar DataFrames
indicadores_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DS}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DS_Dim_socioeconomica
